## 1. SSD (Single Shot MultiBox Detector)

**Contents:**
1. What & Why
2. Architecture
3. Multi-Scale Feature Maps & Default Boxes
4. Matching & Hard Negative Mining
5. Loss
6. torchvision Usage
7. References

---

**SSD** (Liu et al., ECCV 2016) is one of the original **one-stage** ("single shot") detectors. Like YOLO, it predicts boxes and class scores **directly** in a single forward pass — there is **no separate region-proposal stage** — which makes it fast and a good fit for real-time / embedded use.

SSD's two defining ideas are:

1. **Multi-scale prediction:** make detections from **several feature maps of different resolutions**, so small objects are detected on fine (early) maps and large objects on coarse (deep) maps.
2. **Default boxes** (a.k.a. priors / anchors): a fixed set of reference boxes of various scales and aspect ratios tiled over each feature map, against which the network predicts offsets.

This lets a single network handle objects across a wide range of sizes without the image pyramids that earlier detectors needed.

## 2. Architecture

```
Image (e.g. 300x300)
  |
  v
Base network (VGG-16, truncated)  ->  conv4_3 feature map
  |
  v
Extra convolutional layers appended  ->  progressively smaller maps
  (conv4_3, conv7, conv8_2, conv9_2, conv10_2, conv11_2 ...)
  |
  +-- on EACH selected feature map, a small 3x3 conv predictor outputs,
      per default box:
      +--> class scores  (C + 1, includes background)
      +--> 4 box offsets (cx, cy, w, h) relative to that default box
  |
  v
Gather all predictions -> per-class NMS -> detections
```

The base network (originally VGG-16) is followed by a stack of extra conv layers whose spatial resolution shrinks layer by layer. Predictions are read off **multiple** of these layers, so the deeper (smaller) the map, the larger the objects it is responsible for. Everything is convolutional, so the whole thing runs in one pass.

## 3. Multi-Scale Feature Maps & Default Boxes

**Multi-scale feature maps.** A feature map of size `m x n` has `m*n` cells. Early maps (e.g. 38x38) have small receptive fields and high resolution -> good for **small** objects; deep maps (e.g. 3x3, 1x1) have large receptive fields -> good for **large** objects. By predicting from several maps, SSD covers many scales cheaply.

**Default boxes.** At each cell of each chosen feature map, SSD places `k` **default boxes** with preset **scales** and **aspect ratios** (e.g. 1, 2, 3, 1/2, 1/3). The scale assigned to each feature map grows linearly with its depth, so different maps cover different object sizes. For each default box the predictor outputs:

- `C + 1` class confidences (foreground classes plus background), and
- **4 offsets** `(t_cx, t_cy, t_w, t_h)` that refine the default box into the final prediction.

A 300x300 SSD produces roughly **8732** default boxes in total. Conceptually default boxes are the same idea as anchors in Faster R-CNN / RetinaNet, applied across multiple resolutions.

## 4. Matching & Hard Negative Mining

**Matching.** During training each default box is labelled by its **IoU (Jaccard overlap)** with the ground-truth boxes: a default box is **positive** if its IoU with some ground truth exceeds a threshold (0.5), and each ground truth is also matched to its best-overlapping default box so nothing is missed. All remaining default boxes are negatives.

**Hard negative mining.** As with any dense detector, negatives massively outnumber positives (most of the 8732 boxes are background). Training on all of them would let easy backgrounds dominate. SSD's solution: instead of using all negatives, **sort the negatives by their confidence loss (hardest first)** and keep only the top ones so that the **negative:positive ratio is at most 3:1**. This keeps training balanced and stable. (This is SSD's alternative to RetinaNet's focal loss, which solves the same imbalance with a reweighted loss instead of explicit sampling.)

## 5. Loss

The SSD objective is a weighted sum of a localization term and a confidence term, averaged over the `N` matched (positive) default boxes:

```
L = (1 / N) * ( L_conf + alpha * L_loc )
```

- **L_loc** — **Smooth L1** loss between the predicted offsets `(t_cx, t_cy, t_w, t_h)` and the encoded ground-truth box, summed over **positive** boxes only.
- **L_conf** — **softmax cross-entropy** over the `C + 1` classes, computed over the positives **and** the hard-mined negatives (so the kept easy/hard backgrounds get the background label).
- If `N = 0` (no matches), the loss is set to 0.

The weight `alpha` balances the two terms.

## 6. torchvision Usage

torchvision provides the classic 300x300 VGG-16 SSD, plus a lightweight `ssdlite320_mobilenet_v3_large` for mobile.

```python
import torch
from torchvision.models.detection import (
    ssd300_vgg16,
    SSD300_VGG16_Weights,
)

weights = SSD300_VGG16_Weights.DEFAULT
model = ssd300_vgg16(weights=weights)
model.eval()
preprocess = weights.transforms()

from torchvision.io import read_image
img = read_image("image.jpg")          # uint8 CxHxW
batch = [preprocess(img)]

with torch.no_grad():
    out = model(batch)[0]

print(out["boxes"])   # (N, 4)  [x1, y1, x2, y2]
print(out["labels"])  # (N,)
print(out["scores"])  # (N,)

categories = weights.meta["categories"]
keep = out["scores"] > 0.5
for box, label in zip(out["boxes"][keep], out["labels"][keep]):
    print(categories[label], box.tolist())
```

Training mode returns the SSD loss dict:

```python
model.train()
images  = [torch.rand(3, 300, 300)]
targets = [{"boxes": torch.tensor([[10., 20., 200., 250.]]),
            "labels": torch.tensor([1])}]
loss_dict = model(images, targets)   # {'bbox_regression', 'classification'}
loss = sum(loss_dict.values())
loss.backward()
```

For a lighter mobile-friendly model, use `from torchvision.models.detection import ssdlite320_mobilenet_v3_large`.

## 7. References

- Liu, Anguelov, Erhan, Szegedy, Reed, Fu, Berg, *SSD: Single Shot MultiBox Detector*, ECCV 2016 — https://arxiv.org/abs/1512.02325
- torchvision detection models — https://pytorch.org/vision/stable/models.html#object-detection
- `ssd300_vgg16` docs — https://pytorch.org/vision/stable/models/ssd.html